# Unlabelled corpora (DAPT pools)

In [13]:
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "..")
import pandas as pd

from data.twd.loader import load
from utils.corpus_helpers import dedup_key, normalize

## TWD - unfiltered scrape — FOMC registers

In [14]:
twd = pd.read_csv("twd/full_corpus_unfiltered.csv")
print("raw rows:", len(twd))
print(twd["doc_type"].value_counts().to_dict())

twd["sentence"] = twd["sentence"].astype(str).map(normalize)
twd["key"] = twd["sentence"].map(dedup_key)
print("within-source duplicates:", int(twd["key"].duplicated().sum()))
twd = twd.drop_duplicates("key")
print("after dedupe:", len(twd))

raw rows: 179638
{'speech': 107548, 'meeting_minutes': 47340, 'press_conference': 24750}
within-source duplicates: 12744
after dedupe: 166894


## WCB raw corpus — 24 non-US banks

In [15]:
wcb = pd.read_csv(
    "wcb/full_corpus.csv", usecols=["sentence", "bank", "doc_type", "meeting_date"]
)
print("raw rows:", len(wcb), "| banks:", wcb["bank"].nunique())

fed = wcb["bank"].isin(["federal_reserve_system", "fomc"])
print("fed rows dropped:", int(fed.sum()))
wcb = wcb[~fed]

wcb["sentence"] = wcb["sentence"].astype(str).map(normalize)
wcb["key"] = wcb["sentence"].map(dedup_key)
print("within-source duplicates:", int(wcb["key"].duplicated().sum()))
wcb = wcb.drop_duplicates("key")

cross = wcb["key"].isin(set(twd["key"]))
print("cross-source duplicates vs twd:", int(cross.sum()))
wcb = wcb[~cross]
print("after dedupe:", len(wcb))

raw rows: 380200 | banks: 25
fed rows dropped: 47740
within-source duplicates: 35072
cross-source duplicates vs twd: 2
after dedupe: 297386


## Decontamination

In [16]:
train, test = load("benchmark", seed=5768)
labelled = (
    train["sentence"].to_list()
    + test["sentence"].to_list()
    + pd.read_csv("wcb/wcb_train.csv")["sentence"].to_list()
)
labelled_keys = set(pd.Series(labelled, dtype=str).map(normalize).map(dedup_key))
print("labelled key set:", len(labelled_keys))

for name, df in [("twd", twd), ("wcb", wcb)]:
    print(
        f"{name}: {int(df['key'].isin(labelled_keys).sum())} contaminated rows removed"
    )
twd = twd[~twd["key"].isin(labelled_keys)]
wcb = wcb[~wcb["key"].isin(labelled_keys)]
print("clean twd:", len(twd), "| clean wcb:", len(wcb))

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
labelled key set: 25561
twd: 2206 contaminated rows removed
wcb: 22660 contaminated rows removed
clean twd: 164688 | clean wcb: 274726


## Save pools

In [17]:
fomc_pool = twd[["sentence", "doc_type", "meeting_date"]]
global_pool = pd.concat([twd.assign(bank="fomc"), wcb], ignore_index=True)[
    ["sentence", "doc_type", "meeting_date", "bank"]
]

fomc_pool.to_csv("unlabelled_fomc.csv", index=False)
global_pool.to_csv("unlabelled_global.csv", index=False)
print("saved:", len(fomc_pool), "fomc |", len(global_pool), "global")

# Colab reads the pools from Drive; skipped when Drive is not mounted
drive = "G:/My Drive/thesis"
for f in ("unlabelled_fomc.csv", "unlabelled_global.csv"):
    if Path(drive).is_dir():
        shutil.copy2(f, f"{drive}/{f}")
        print("copied to Drive:", f)

saved: 164688 fomc | 439414 global
copied to Drive: unlabelled_fomc.csv
copied to Drive: unlabelled_global.csv


## Summary

In [18]:
print("FOMC pool:", len(fomc_pool))
print(fomc_pool["doc_type"].value_counts().to_dict())
print("Global pool:", len(global_pool), "| non-FOMC increment:", len(wcb))
print("banks in global pool:", global_pool["bank"].nunique())

FOMC pool: 164688
{'speech': 100169, 'meeting_minutes': 41155, 'press_conference': 23364}
Global pool: 439414 | non-FOMC increment: 274726
banks in global pool: 25
